In [1]:
!pip install -q --upgrade diffusers
# Then restart the kernel and rerun from the pipe loading cell

In [2]:
import torch, os
from diffusers import StableVideoDiffusionPipeline
from diffusers.utils import load_image, export_to_video
from IPython.display import Video, display
from PIL import Image

torch.cuda.empty_cache()
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# SVD is ~7GB and well-tested on Kaggle T4
pipe = StableVideoDiffusionPipeline.from_pretrained(
    "stabilityai/stable-video-diffusion-img2vid-xt",
    torch_dtype=torch.float16,
    variant="fp16",
)
pipe.enable_model_cpu_offload()
pipe.unet.enable_forward_chunking()  # saves VRAM during attention

# SVD requires exactly 1024x576 input
image = load_image(
    "https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/diffusers/guitar-man.png"
).resize((1024, 576))

frames = pipe(
    image,
    num_frames=25,
    num_inference_steps=25,
    decode_chunk_size=4,      # decode 4 frames at a time to save VRAM
    motion_bucket_id=24,     # 1-255: higher = more motion
    noise_aug_strength=0.02,
    generator=torch.Generator("cpu").manual_seed(666),
).frames[0]

export_to_video(frames, "/kaggle/working/output.mp4", fps=7)
print("Done!")
display(Video("/kaggle/working/output.mp4", embed=True, width=640))

Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.


Loading pipeline components...:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/520 [00:00<?, ?it/s]

  0%|          | 0/25 [00:00<?, ?it/s]

Done!


In [ ]:
# Option B#
# same picture, different seed, merge video, watermark

In [3]:
import torch, os
from diffusers import StableVideoDiffusionPipeline
from diffusers.utils import load_image, export_to_video
from IPython.display import Video, display
from PIL import Image, ImageDraw, ImageFont  # ← added ImageDraw, ImageFont

torch.cuda.empty_cache()
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

pipe = StableVideoDiffusionPipeline.from_pretrained(
    "stabilityai/stable-video-diffusion-img2vid-xt",
    torch_dtype=torch.float16,
    variant="fp16",
)
pipe.enable_model_cpu_offload()
pipe.unet.enable_forward_chunking()

image = load_image(
    "https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/diffusers/guitar-man.png"
).resize((1024, 576))

# ── only new additions below ──────────────────────────────────────────

SEEDS = [42, 123, 999]          # ← one video will be generated per seed
all_frames = []                 # ← collects frames from every iteration

def add_watermark(frames, seed):
    """Burn the seed number into the bottom-right corner of every frame."""
    watermarked = []
    for frame in frames:
        frame = frame.copy()
        draw = ImageDraw.Draw(frame)
        text = f"seed={seed}"
        draw.text((900, 545), text, fill=(255, 255, 255))   # white text
        watermarked.append(frame)
    return watermarked

for seed in SEEDS:
    print(f"Generating video for seed={seed}...")
    torch.cuda.empty_cache()                                # free VRAM between runs

    frames = pipe(
        image,
        num_frames=25,
        num_inference_steps=25,
        decode_chunk_size=4,
        motion_bucket_id=127,
        noise_aug_strength=0.02,
        generator=torch.Generator("cpu").manual_seed(seed), # ← seed changes each loop
    ).frames[0]

    frames = add_watermark(frames, seed)                    # ← stamp the seed
    all_frames.extend(frames)                               # ← append to merged list
    print(f"  seed={seed} done ({len(frames)} frames collected so far: {len(all_frames)})")

# ── export the single merged video ───────────────────────────────────
export_to_video(all_frames, "/kaggle/working/output_merged.mp4", fps=7)
print(f"\nDone! Total frames: {len(all_frames)} → ~{len(all_frames)/7:.1f}s video")
display(Video("/kaggle/working/output_merged.mp4", embed=True, width=640))

Loading pipeline components...:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/520 [00:00<?, ?it/s]

Generating video for seed=42...


  0%|          | 0/25 [00:00<?, ?it/s]

  seed=42 done (25 frames collected so far: 25)
Generating video for seed=123...


  0%|          | 0/25 [00:00<?, ?it/s]

  seed=123 done (25 frames collected so far: 50)
Generating video for seed=999...


  0%|          | 0/25 [00:00<?, ?it/s]

  seed=999 done (25 frames collected so far: 75)

Done! Total frames: 75 → ~10.7s video
